# Phase 14 — Model Interpretability and Explainability

This notebook reviews the saved validation-only explanations for Logistic Regression, TextCNN, vanilla RNN, LSTM, and GRU. It does not train or tune models. By default it loads the latest completed Phase 14 output folder and does not load the IMDb dataset or official test split. If `RUN_ANALYSIS` is enabled, the shared package runner reconstructs the deterministic validation split, verifies the saved IDs and fingerprints, then loads the five frozen models for inference.

The neural occlusion score is the original positive-class probability minus the masked positive-class probability. A positive score means the window supported the positive class for that example. These are sensitivity analyses, not causal claims.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

RUN_ANALYSIS = False  # Set True only to create a new validation-only output run.
if RUN_ANALYSIS:
    from deeptext.experiments.interpretability import run_interpretability
    result = run_interpretability()
    output_directory = Path(result['output_directory'])
else:
    runs = sorted(Path('outputs/interpretability').glob('imdb-interpretability-*'))
    if not runs:
        raise FileNotFoundError('No Phase 14 output exists. Set RUN_ANALYSIS=True to create it.')
    output_directory = runs[-1]

metadata = json.loads((output_directory / 'metadata' / 'run.json').read_text(encoding='utf-8'))
assert metadata['validation_only'] and not metadata['official_test_split_loaded']
assert not metadata['model_training_performed'] and not metadata['checkpoint_files_modified']
print(f'Output: {output_directory}')
print(f"Validation rows: {metadata['split']['validation_rows']:,}; selected examples: {metadata['selected_example_count']}")
display(pd.DataFrame([{'model': name, **details} for name, details in metadata['models'].items()]))

In [ ]:
examples = json.loads((output_directory / 'cross_model' / 'validation_examples_and_predictions.json').read_text(encoding='utf-8'))
display(pd.DataFrame([{key: value for key, value in row.items() if key != 'review_text'} for row in examples]))
for row in examples:
    print(f"{row['review_id']} | true={row['true_label']} | tags={', '.join(row['selection_tags'])}")
    print(row['review_text'][:500].replace('\n', ' ') + ('…' if len(row['review_text']) > 500 else ''))
    print()

In [ ]:
positive_features = pd.read_csv(output_directory / 'logistic_regression' / 'global_top_positive.csv')
negative_features = pd.read_csv(output_directory / 'logistic_regression' / 'global_top_negative.csv')
display(positive_features.head(20))
display(negative_features.head(20))
contributions = pd.read_csv(output_directory / 'logistic_regression' / 'review_feature_contributions.csv')
display(contributions.groupby('review_id', as_index=False).agg(decision_function=('decision_function', 'first'), intercept=('intercept', 'first')))
print('All nonzero TF-IDF terms plus the intercept were checked against the decision function:', metadata['models']['logistic_regression']['all_reconstructions_validated'])

In [ ]:
for model_name in ('textcnn', 'rnn', 'lstm', 'gru'):
    occlusion_path = output_directory / model_name / 'token_window_occlusion.csv'
    occlusion = pd.read_csv(occlusion_path)
    print(f'\n{model_name.upper()} — largest positive-class probability drops')
    display(occlusion.sort_values('positive_probability_drop', ascending=False).groupby('review_id', as_index=False).head(8)[['review_id', 'tokens', 'positive_probability_drop']])
    state_path = output_directory / model_name / 'hidden_state_evolution.csv'
    if state_path.exists():
        display(pd.read_csv(state_path).groupby('review_id', as_index=False).agg(token_positions=('position', 'count'), mean_hidden_l2=('hidden_l2_norm', 'mean'), max_hidden_l2=('hidden_l2_norm', 'max')))
    gate_path = output_directory / model_name / 'gate_activations.csv'
    if gate_path.exists():
        gates = pd.read_csv(gate_path)
        print('Mean gate activations by token position:')
        display(gates.groupby(['gate', 'position'], as_index=False).mean(numeric_only=True).head(24))

pairwise = pd.read_csv(output_directory / 'cross_model' / 'pairwise_explanation_comparison.csv')
print('Pairwise disagreements with both models’ local evidence:')
display(pairwise.loc[~pairwise.predictions_agree].head(30))

In [ ]:
for plot_path in sorted((output_directory / 'plots').glob('*.png')):
    image = plt.imread(plot_path)
    plt.figure(figsize=(12, 5))
    plt.imshow(image)
    plt.axis('off')
    plt.title(plot_path.stem.replace('_', ' ').title())
    plt.show()
    plt.close()

## Final observations and limits

The explanations use different quantities: Logistic Regression feature contributions are additive decision-function terms; TextCNN activations are filter responses; recurrent traces show states and gates; occlusion reports changes in positive-class probability. Their magnitudes should not be ranked against each other. A large filter activation or gate value is not evidence of a human-like concept. The selected reviews cover prediction cases and are not an unbiased sample of the full validation set.

Run metadata records the deterministic training/validation fingerprints, selected review IDs, artifact hashes, model/checkpoint preservation flags, and that the official test split was not loaded. Logistic Regression contributions reconstruct its decision function. LSTM and GRU gate tables are present only when analysis-only equations reproduce native hidden states and logits within the recorded tolerance. Probability scores are uncalibrated and should not be read as certainty.